In [2]:
"""""
Helper Utilities for Prompt Engineering Challenge

This module provides utility functions for calling LLMs with structured output.
"""

import os
import json
from typing import Dict, Any, Optional
from openai import OpenAI


def call_llm_structured(prompt: str) -> Dict[str, Any]:
    """
    Call OpenAI API with structured JSON output.

    This is the core helper function students use to implement their prompts.

    Args:
        prompt: The prompt to send to the LLM

    Returns:
        Parsed JSON response as a dictionary

    Raises:
        ValueError: If OPENAI_API_KEY is not set
        Exception: For other API errors
    """
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        raise ValueError(
            "OPENAI_API_KEY environment variable not set. "
            "Please set it to use real LLM calls."
        )

    client = OpenAI(api_key=api_key)

    try:
        # Use JSON mode for structured output
        response = client.chat.completions.create(
            model="gpt-5-mini",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"}
        )

        # Parse JSON response
        content = response.choices[0].message.content
        return json.loads(content)

    except json.JSONDecodeError as e:
        raise Exception(f"LLM returned invalid JSON: {e}\nResponse: {content}")
    except Exception as e:
        raise Exception(f"Error calling OpenAI API: {e}")


def format_json_pretty(data: Dict[str, Any]) -> str:
    """
    Format JSON data for human-readable display.

    Args:
        data: Dictionary to format

    Returns:
        Pretty-printed JSON string
    """
    return json.dumps(data, indent=2, default=str)


def validate_required_fields(data: Dict[str, Any], required_fields: list) -> bool:
    """
    Validate that all required fields are present in data.

    Args:
        data: Dictionary to validate
        required_fields: List of required field names

    Returns:
        True if all fields present, False otherwise
    """
    return all(field in data for field in required_fields)


# Example usage
if __name__ == "__main__":
    # Test with a simple prompt
    test_prompt = '''
    Classify this code issue. Respond with JSON:
    {
      "category": "security",
      "severity": "critical",
      "description": "SQL injection vulnerability"
    }
    '''

    try:
        result = call_llm_structured(test_prompt)
        print("Result:")
        print(format_json_pretty(result))
    except Exception as e:
        print(f"Error: {e}")


Error: OPENAI_API_KEY environment variable not set. Please set it to use real LLM calls.


In [ ]:
"""
Technical Debt Classifier

This module classifies technical debt using LLM with structured output.

You need to implement classification using prompt engineering techniques:
- Clear role assignment
- Structured JSON output
- Few-shot examples
- Context management
"""

import json
from dataclasses import dataclass, asdict
from typing import Dict, Any
# from helpers import call_llm_structured


@dataclass
class DebtClassification:
    """Classification result for a technical debt item."""
    category: str  # 'security', 'performance', 'maintainability', 'reliability'
    severity: str  # 'critical', 'high', 'medium', 'low'
    description: str
    fix_effort_days: float
    reasoning: str


def classify_debt(code_snippet: str, file_path: str, pattern_type: str) -> DebtClassification:
    """
    Classify technical debt using LLM with structured output.

    Your implementation should:
    1. Write a clear prompt with role assignment (e.g., "You are a senior software engineer...")
    2. Provide classification criteria:
       - Categories: security, performance, maintainability, reliability
       - Severity levels: critical, high, medium, low
       - Effort estimation guidelines
    3. Include 2-3 few-shot examples showing different classifications
    4. Request structured JSON output
    5. Handle the LLM response and create DebtClassification object

    Args:
        code_snippet: The code with the issue
        file_path: Path to the file
        pattern_type: Type of pattern detected (e.g., 'sql_injection')

    Returns:
        DebtClassification object with structured classification

    Hints:
    - Start with role assignment and task description
    - Show the issue details (file path, pattern type, code snippet)
    - Define clear categories and severity levels with descriptions
    - Provide concrete examples (e.g., SQL injection → security/critical/2 days)
    - Request JSON: {"category": "...", "severity": "...", "description": "...", "fix_effort_days": X, "reasoning": "..."}
    - Use call_llm_structured() from helpers
    - Include error handling with fallback classification
    """

    prompt = """
    You are a senior software engineer tasked with classifying technical debt in code.

    You will receive:
    - A code snippet
    - The file path
    - A detected pattern type

    Your task is to classify the technical debt according to:
    - Categories: security, performance, maintainability, reliability

    Severity definitions:
    - Critical: Immediate risk to security, system stability, or data integrity. Requires urgent remediation.
    - High: Significant impact on functionality, performance, or reliability, but not immediately catastrophic.
    - Medium: Noticeable maintainability, reliability, or performance issue that should be addressed in normal development.
    - Low: Minor issue with limited impact; primarily affects readability, maintainability, or future development.

    Effort estimation guidelines (fix_effort_days):
    - 0.5: Simple localized change with minimal testing.
    - 1.0: Small refactoring or straightforward fix affecting a single module.
    - 2.0: Moderate refactoring requiring updates across multiple functions or files and additional testing.
    - 3.0+: Complex redesign, architectural changes, or fixes involving multiple components.

    Classification rules:
    1. Determine the primary category (security, performance, maintainability, or reliability).
    2. Assign severity using the severity definitions above.
    3. Estimate fix_effort_days using the effort guidelines.
    4. Base the classification only on the provided code snippet and pattern.
    5. If multiple categories apply, choose the one representing the primary technical debt.

    Example 1:
    File: src/database.py
    Pattern: sql_injection
    Code Snippet:
    query = f"SELECT * FROM users WHERE id = \'{{user_id}}\'"

    Classification:
    {{
        "category": "security",
        "severity": "critical",
        "description": "SQL injection vulnerability caused by unsanitized user input.",
        "fix_effort_days": 2.0,
        "reasoning": "User input is directly interpolated into the SQL statement, allowing arbitrary SQL execution. Parameterized queries eliminate this risk."
    }}

    Example 2:
    File: src/utils.py
    Pattern: nested_loops
    Code Snippet:
    for i in range(len(data)):
        for j in range(len(data)):
            process(data[i], data[j])

    Classification:
    {{
        "category": "performance",
        "severity": "high",
        "description": "Quadratic time complexity due to nested iteration.",
        "fix_effort_days": 1.5,
        "reasoning": "The O(n²) algorithm may become a bottleneck for large datasets. A more efficient algorithm or indexing strategy should be considered."
    }}

    Example 3:
    File: src/helpers.py
    Pattern: duplicated_code
    Code Snippet:
    def calculate_total(...):
        ...

    def calculate_total_backup(...):
        ...

    Classification:
    {{
        "category": "maintainability",
        "severity": "medium",
        "description": "Duplicated business logic increases maintenance cost.",
        "fix_effort_days": 1.0,
        "reasoning": "Changes must be applied in multiple locations, increasing the likelihood of inconsistencies. Extracting shared logic into a reusable function improves maintainability."
    }}

    Example 4:
    File: src/service.py
    Pattern: broad_exception
    Code Snippet:
    try:
        process_request()
    except Exception:
        pass

    Classification:
    {{
        "category": "reliability",
        "severity": "low",
        "description": "Broad exception handling suppresses unexpected errors.",
        "fix_effort_days": 0.5,
        "reasoning": "Silently ignoring exceptions can hide failures and complicate debugging. Catch specific exceptions and log unexpected errors."
    }}

    Now classify the following issue:

    File: {file_path}
    Pattern: {pattern_type}
    Code Snippet:
    {code_snippet}

    Respond ONLY with valid JSON using this schema:
    {{
        "category": "...",
        "severity": "...",
        "description": "...",
        "fix_effort_days": X,
        "reasoning": "..."
    }}
    """

    response = call_llm_structured(
        prompt.format(
            file_path=file_path,
            pattern_type=pattern_type,
            code_snippet=code_snippet
        ))

    try:
        classification = DebtClassification(
            category=response.get('category', 'unknown'),
            severity=response.get('severity', 'unknown'),
            description=response.get('description', ''),
            fix_effort_days=float(response.get('fix_effort_days', 0.0)),
            reasoning=response.get('reasoning', '')
        )
        return classification
    except Exception as e:
        print(f"Error parsing LLM response: {e}. Response: {response}")
        # Fallback classification
        return DebtClassification(
            category='unknown',
            severity='unknown',
            description='Failed to classify due to LLM error.',
            fix_effort_days=0.0,
            reasoning='LLM response could not be parsed.'
        )


def classify_multiple(issues: list) -> list[DebtClassification]:
    """
    Classify multiple debt items.

    Args:
        issues: List of dicts with 'code_snippet', 'file_path', 'pattern_type'

    Returns:
        List of DebtClassification objects
    """
    classifications = []

    for issue in issues:
        classification = classify_debt(
            code_snippet=issue.get('code_snippet', ''),
            file_path=issue.get('file_path', ''),
            pattern_type=issue.get('pattern_type', 'unknown')
        )
        classifications.append(classification)

    return classifications


def get_classification_summary(classifications: list[DebtClassification]) -> Dict[str, Any]:
    """
    Generate summary statistics from classifications.

    Args:
        classifications: List of DebtClassification objects

    Returns:
        Dictionary with summary statistics
    """
    if not classifications:
        return {
            'total_items': 0,
            'by_category': {},
            'by_severity': {},
            'total_effort_days': 0.0,
            'critical_count': 0
        }

    # Count by category
    by_category = {}
    for c in classifications:
        by_category[c.category] = by_category.get(c.category, 0) + 1

    # Count by severity
    by_severity = {}
    for c in classifications:
        by_severity[c.severity] = by_severity.get(c.severity, 0) + 1

    # Total effort
    total_effort = sum(c.fix_effort_days for c in classifications)

    # Critical count
    critical_count = sum(1 for c in classifications if c.severity == 'critical')

    return {
        'total_items': len(classifications),
        'by_category': by_category,
        'by_severity': by_severity,
        'total_effort_days': round(total_effort, 1),
        'critical_count': critical_count
    }


# Example usage
if __name__ == "__main__":
    # Test with a sample issue
    sample_code = '''
    query = f"SELECT * FROM users WHERE id = '{user_id}'"
    '''

    classification = classify_debt(
        code_snippet=sample_code,
        file_path="src/database.py",
        pattern_type="sql_injection"
    )

    print("Classification Result:")
    print(json.dumps(asdict(classification), indent=2))


In [ ]:
"""
Priority Ranker - Sprint Planning with LLM

This module ranks technical debt items and creates sprint plans using LLM reasoning.

You need to implement sprint planning using advanced prompt engineering techniques:
- Multi-item context formatting
- Chain of thought reasoning
- Criteria-based decision making
- Structured list output
"""

import json
from dataclasses import dataclass, asdict
from typing import List, Dict, Any
# from helpers import call_llm_structured


@dataclass
class RankedItem:
    """A ranked technical debt item."""
    item_id: int
    category: str
    severity: str
    description: str
    fix_effort_days: float
    priority_score: float  # 0-10
    reasoning: str


@dataclass
class SprintPlan:
    """A complete sprint plan with ranked items."""
    strategy_name: str
    ranked_items: List[RankedItem]
    sprint_plan: List[RankedItem]  # Items that fit in capacity
    total_effort_days: float
    capacity_utilization: float  # 0.0-1.0
    explanation: str


def rank_debt_items(debt_items: List[Dict[str, Any]], team_capacity_days: float = 10.0) -> SprintPlan:
    """
    Rank technical debt items and create sprint plan.

    Your implementation should:
    1. Format debt items as clear, structured context for the LLM
    2. Provide explicit prioritization criteria (risk, effort vs value, urgency)
    3. Use chain of thought reasoning to guide the LLM through decision-making
    4. Request structured JSON output with ranked items and sprint selection

    Args:
        debt_items: List of classified debt items (dicts with category, severity, description, fix_effort_days)
        team_capacity_days: Available team capacity in days (default: 10.0)

    Returns:
        SprintPlan with ranked items and sprint selection

    Hints:
    - Format each item with: Item ID, Category, Severity, Description, Fix Effort
    - Define clear prioritization criteria (e.g., Risk Impact, Effort vs Value, Dependencies, Urgency)
    - Include chain of thought instructions (e.g., "Think step by step: 1. Identify critical issues...")
    - Provide 1-2 concrete examples showing input items → output rankings
    - Request JSON: {"ranked_items": [...], "sprint_selection": [...], "total_effort": X, "explanation": "..."}
    - Each ranked item should include: item_id, priority_score (0-10), reasoning
    """

    items = []
    for idx, item in enumerate(debt_items, start=1):
        items.append(f"Item {idx}:\n"
                     f"- Category: {item.get('category', 'unknown')}\n"
                     f"- Severity: {item.get('severity', 'unknown')}\n"
                     f"- Description: {item.get('description', '')}\n"
                     f"- Fix Effort: {item.get('fix_effort_days', 0.0)} days\n")

    items_text = "\n".join(items)
    prompt = f"""You are a senior software engineer tasked with ranking technical debt items and creating a sprint plan.
    You have a team capacity of {team_capacity_days} days.
    Here are the technical debt items to consider:

    {items_text}

    Please rank the items based on the following criteria:
    1. Risk Impact: How critical is the issue to system security, performance, or reliability?
    2. Effort vs Value: How much effort is required to fix the issue compared to the value of resolving it?
    3. Dependencies: Are there any dependencies that affect the prioritization?

    Evaluate the items using the criteria above and return only the requested JSON.

    Example 1:
    Input Items:
    Item 1: Security, Critical, SQL injection vulnerability, 2 days
    Item 2: Maintainability, Low, TODO comment, 0.5 days

    Output:
    {{
      "ranked_items": [
        {{
          "item_id": 1,
          "priority_score": 10,
          "reasoning": "SQL injection is a critical security vulnerability that must be fixed immediately."
        }},
        {{
          "item_id": 2,
          "priority_score": 2,
          "reasoning": "A TODO comment is low priority and can be deferred."
        }}
      ],
      "sprint_selection": [
        {{
          "item_id": 1,
          "priority_score": 10,
          "reasoning": "Selected for sprint due to critical security risk."
        }}
      ],
      "total_effort": 2,
      "explanation": "The sprint plan focuses on addressing the most critical security issue within the available capacity."
    }}

    Example 2:
    Input Items:
    Item 1: Performance, High, Nested loops causing O(n^2) complexity, 1.5 days
    Item 2: Reliability, Medium, Unhandled exception in API, 1 day

    Output:
    {{
      "ranked_items": [
        {{
          "item_id": 1,
          "priority_score": 8,
          "reasoning": "Performance issue can lead to slow response times, affecting user experience."
        }},
        {{
          "item_id": 2,
          "priority_score": 6,
          "reasoning": "Unhandled exceptions can cause API failures, but the impact is less severe than performance degradation."
        }}
      ],
      "sprint_selection": [
        {{
          "item_id": 1,
          "priority_score": 8,
          "reasoning": "Selected for sprint due to high impact on performance."
        }},
        {{
          "item_id": 2,
          "priority_score": 6,
          "reasoning": "Selected for sprint as it can be fixed within capacity and improves reliability."
        }}
      ],
      "total_effort": 2.5,
      "explanation": "The sprint plan addresses both performance and reliability issues within the available capacity."
    }}

    Provide your response in the following JSON format:
    {{
      "ranked_items": [
        {{
          "item_id": 1,
          "priority_score": 0,
          "reasoning": "..."
        }}
      ],
      "sprint_selection": [
        {{
          "item_id": 1,
          "priority_score": 0,
          "reasoning": "..."
        }}
      ],
      "total_effort": 0,
      "explanation": "..."
    }}"""

    result = call_llm_structured(prompt)

    rankedItems = []
    for item in result.get('ranked_items', []):
        ranked_item = RankedItem(
            item_id=item.get('item_id', 0),
            category=debt_items[item.get('item_id', 1) - 1].get('category', 'unknown'),
            severity=debt_items[item.get('item_id', 1) - 1].get('severity', 'unknown'),
            description=debt_items[item.get('item_id', 1) - 1].get('description', ''),
            fix_effort_days=debt_items[item.get('item_id', 1) - 1].get('fix_effort_days', 0.0),
            priority_score=item.get('priority_score', 0.0),
            reasoning=item.get('reasoning', '')
        )
        rankedItems.append(ranked_item)

    sprint_selection = []
    for item in rankedItems:
        if sum(i.fix_effort_days for i in sprint_selection) + item.fix_effort_days <= team_capacity_days:
            sprint_selection.append(item)
        else:
            break  # Stop adding items once capacity is exceeded

    total_effort = sum(item.fix_effort_days for item in sprint_selection)
    capacity_utilization = total_effort / team_capacity_days if team_capacity_days > 0 else 0.0

    return SprintPlan(
        strategy_name="LLM-Based Prioritization",
        ranked_items=rankedItems,
        sprint_plan=sprint_selection,
        total_effort_days=total_effort,
        capacity_utilization=capacity_utilization,
        explanation=result.get('explanation', 'No explanation provided by LLM.')
    )


# Example usage
if __name__ == "__main__":
    # Test with sample items
    sample_items = [
        {
            'category': 'security',
            'severity': 'critical',
            'description': 'SQL injection vulnerability',
            'fix_effort_days': 2.0
        },
        {
            'category': 'maintainability',
            'severity': 'low',
            'description': 'TODO comment',
            'fix_effort_days': 0.5
        },
        {
            'category': 'security',
            'severity': 'critical',
            'description': 'Hardcoded credentials',
            'fix_effort_days': 3.0
        }
    ]

    plan = rank_debt_items(sample_items, team_capacity_days=10.0)

    print("Sprint Plan:")
    print(json.dumps(asdict(plan), indent=2, default=str))


ValueError: OPENAI_API_KEY environment variable not set. Please set it to use real LLM calls.

In [ ]:
"""
Quality Validator - LLM-as-Judge Pattern

This module validates sprint plan quality using LLM as a judge.

You need to implement quality validation using the LLM-as-judge pattern:
- LLM-as-judge evaluation
- Clear rubric definition
- Multi-dimensional scoring
- Actionable feedback generation
"""

import json
from dataclasses import dataclass, asdict
from typing import Dict, Any, List
# from helpers import call_llm_structured


@dataclass
class QualityScores:
    """Multi-dimensional quality scores."""
    feasibility: float  # 0-10: Can team realistically complete this?
    completeness: float  # 0-10: Are all critical issues addressed?
    risk_coverage: float  # 0-10: Are high-risk items prioritized?
    balance: float  # 0-10: Good mix of quick wins and important work?
    overall: float  # 0-10: Average of all dimensions
    passed: bool  # True if overall >= 7.0
    feedback: str


def validate_sprint_quality(sprint_plan: Dict[str, Any], team_capacity_days: float = 10.0) -> QualityScores:
    """
    Validate sprint plan quality using LLM-as-judge.

    Your implementation should:
    1. Create a comprehensive evaluation rubric with 4 dimensions:
       - Feasibility: Can the team complete this within capacity?
       - Completeness: Are critical issues addressed?
       - Risk Coverage: Are high-risk items prioritized?
       - Balance: Good mix of quick wins and important work?

    2. Build a judge prompt that includes:
       - Sprint plan details (items, effort, capacity utilization)
       - Clear scoring rubric for each dimension (0-10 scale)
       - 2-3 concrete examples showing different quality levels
       - Request structured JSON output with scores and feedback

    3. Call the LLM using call_llm_structured() from helpers

    4. Parse the response and calculate:
       - Individual dimension scores (0-10)
       - Overall score (average of 4 dimensions)
       - Pass/fail threshold (overall >= 7.0)
       - Actionable feedback (2-3 sentences)

    5. Include error handling with fallback to rule-based validation

    Args:
        sprint_plan: Sprint plan dict with 'sprint_plan', 'total_effort_days', 'explanation'
        team_capacity_days: Team capacity in days (default: 10.0)

    Returns:
        QualityScores with multi-dimensional evaluation

    Hints:
    - Format sprint items clearly for the judge (severity, category, effort)
    - Calculate capacity utilization percentage
    - Use concrete scoring bands in your rubric (e.g., 9-10: excellent, 7-8: good, etc.)
    - Provide diverse examples (excellent plan, overcommitted, ignores critical issues)
    - Request JSON format: {"feasibility": X, "completeness": Y, "risk_coverage": Z, "balance": W, "feedback": "..."}
    """

    sprint_items = sprint_plan.get('sprint_plan', [])
    total_effort = sprint_plan.get('total_effort_days', 0.0)

    formatted_items = []
    for item in sprint_items:
        formatted_items.append(f"[{str(item.get('severity', 'unknown')).upper()}] {item.get('category', 'unknown')}: {item.get('description', '')} ({item.get('fix_effort_days', 0.0)} days)")

    capacity_utilization = total_effort / team_capacity_days * 100

    prompt = f"""
    You are a senior Software Quality Assurance (QA) engineer responsible for reviewing sprint plans for technical debt remediation.

    Your objective is to evaluate whether the proposed sprint is realistic, well-prioritized, and provides the highest value while respecting the available team capacity.

    ## Sprint Information

    Team Capacity: {team_capacity_days:.1f} days
    Planned Effort: {total_effort:.1f} days
    Capacity Utilization: {capacity_utilization:.1f}%

    Sprint Items:
    {chr(10).join(formatted_items)}

    ---

    Evaluate the sprint using the following dimensions.

    ### 1. Feasibility
    Evaluate whether the sprint is realistically achievable.

    Consider:
    - Capacity utilization
    - Total planned effort
    - Risk of overcommitment

    Scoring:
    - 9–10: Fits comfortably within capacity (70–95%)
    - 7–8: Near capacity but realistic (95–100%)
    - 4–6: Slightly overloaded or inefficiently underutilized
    - 0–3: Clearly unrealistic (>100%) or severely underutilized (<50%)

    ---

    ### 2. Completeness
    Evaluate whether the sprint includes the most important technical debt.

    Consider:
    - Critical issues included
    - High severity issues addressed
    - Missing high-impact work

    Scoring:
    - 9–10: All critical issues included
    - 7–8: Most critical issues included
    - 4–6: Some important work missing
    - 0–3: Critical work omitted

    ---

    ### 3. Risk Coverage
    Evaluate whether the sprint prioritizes the highest-risk issues.

    Consider:
    - Security
    - Reliability
    - Performance
    - Business impact

    Scoring:
    - 9–10: Highest-risk items prioritized first
    - 7–8: Most high-risk work prioritized
    - 4–6: Mixed prioritization
    - 0–3: Low-risk work prioritized over critical issues

    ---

    ### 4. Balance
    Evaluate whether the sprint contains an effective mix of work.

    Consider:
    - Quick wins
    - High-value improvements
    - Long-term maintainability
    - Variety of issue categories

    Scoring:
    - 9–10: Excellent balance
    - 7–8: Good balance with minor bias
    - 4–6: Noticeable imbalance
    - 0–3: Poorly balanced sprint

    ---

    Evaluation Rules

    - Base your evaluation ONLY on the provided sprint items.
    - Do not invent missing work.
    - Be objective and consistent.
    - Penalize overloaded sprints.
    - Prioritize critical and high-risk issues over low-severity work.
    - Explain deductions clearly.
    - Return ONLY valid JSON.

    Example 1

    Input

    Capacity Utilization: 50%

    Items
    - [CRITICAL] Security – SQL injection (2 days)
    - [CRITICAL] Security – Hardcoded credentials (3 days)

    Output

    {{
    "feasibility": 10,
    "completeness": 10,
    "risk_coverage": 10,
    "balance": 8,
    "feedback": "The sprint is well within capacity and addresses all critical security issues. Although it focuses heavily on security, this prioritization is appropriate given the severity."
    }}

    Example 2

    Input

    Capacity Utilization: 120%

    Items
    - [CRITICAL] Security – SQL injection (2 days)
    - [CRITICAL] Security – Hardcoded credentials (3 days)
    - [LOW] Performance – Inefficient query (1 day)
    - [LOW] Maintainability – TODO comments (0.5 days)

    Output

    {{
    "feasibility": 2,
    "completeness": 9,
    "risk_coverage": 9,
    "balance": 4,
    "feedback": "The sprint exceeds team capacity, making it unrealistic. While critical issues are included, low-priority work should be deferred to reduce overcommitment."
    }}

    Example 3

    Input

    Capacity Utilization: 90%

    Items
    - [LOW] Maintainability – TODO comments (0.5 days)
    - [LOW] Performance – Logging cleanup (0.5 days)

    Output

    {{
    "feasibility": 9,
    "completeness": 2,
    "risk_coverage": 1,
    "balance": 3,
    "feedback": "The sprint is feasible but poorly prioritized. It focuses on low-impact improvements while ignoring higher-risk technical debt."
    }}

    Return ONLY valid JSON using this schema:

    {{
    "feasibility": <0-10>,
    "completeness": <0-10>,
    "risk_coverage": <0-10>,
    "balance": <0-10>,
    "feedback": "<concise explanation>"
    }}
    """

    result = call_llm_structured(prompt)

    feasibility = result.get('feasibility', 0.0)
    completeness = result.get('completeness', 0.0)
    risk_coverage = result.get('risk_coverage', 0.0)
    balance = result.get('balance', 0.0)
    feedback = result.get('feedback', '')

    overall = (feasibility + completeness + risk_coverage + balance) / 4

    return QualityScores(
        feasibility=feasibility,
        completeness=completeness,
        risk_coverage=risk_coverage,
        balance=balance,
        overall=overall,
        passed=overall >= 7.0,
        feedback=feedback
    )


# Example usage
if __name__ == "__main__":
    # Test with sample sprint plan
    sample_plan = {
        'sprint_plan': [
            {
                'category': 'security',
                'severity': 'critical',
                'description': 'SQL injection vulnerability',
                'fix_effort_days': 2.0
            },
            {
                'category': 'security',
                'severity': 'critical',
                'description': 'Hardcoded credentials',
                'fix_effort_days': 3.0
            }
        ],
        'total_effort_days': 5.0,
        'explanation': 'Prioritized critical security issues'
    }

    scores = validate_sprint_quality(sample_plan, team_capacity_days=10.0)

    print("Quality Evaluation:")
    print(json.dumps(asdict(scores), indent=2))